# Genre-Extraction on Kaggle — reproduction (Obj. 1) + improved model & tuning (Obj. 2)
**Before running:** *Add Data* → "GTZAN Dataset - Music Genre Classification"; Accelerator → GPU.
Sections: 0 setup · 1 write source files · 2 smoke test · 3 reproduce paper · 4 improved models · 5 tuning · 6 results/zip.

In [ ]:
import os, sys, subprocess
os.makedirs("/kaggle/working/genre-extraction-repro/src/genre", exist_ok=True)
os.makedirs("/kaggle/working/genre-extraction-repro/scripts", exist_ok=True)
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
try:
    import optuna
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"])   # needs Internet ON

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/__init__.py
"""Genre-Extraction reproduction + improvements (TensorFlow 2 / Keras)."""

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/config.py
GENRES = ["classical", "jazz", "metal", "pop"]   # same 4 classes / same label order as the original repo
SEED = 42
# colours used by the original testsoftmaxautoencoder.py for PCA plots (index == label)
GENRE_COLORS = ["navy", "green", "darkorange", "purple"]

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/utils.py
import os
import random
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split

from .config import GENRES


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except ImportError:
        pass


def find_gtzan_root(explicit=None, genres=GENRES):
    """Return the folder that directly contains <genre>/ sub-folders full of .wav files (any nesting depth)."""
    def ok(d):
        # folders must exist AND contain .wav files (this dataset also ships Data/images_original full of PNGs)
        return all(os.path.isdir(os.path.join(d, g)) and
                   any(f.lower().endswith(".wav") for f in os.listdir(os.path.join(d, g))) for g in genres)

    if explicit and explicit != "auto":
        if ok(explicit):
            return explicit
        raise FileNotFoundError(f"{explicit} does not contain the folders {genres}")
    seen = []
    for base in ("/kaggle/input", "./genres", "./data", "."):
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, _ in os.walk(base):
            if ok(dirpath):                      # check BEFORE anything else; no depth limit
                return dirpath
            if len(seen) < 25 and dirpath.count(os.sep) <= 6:
                seen.append(dirpath)
            if "outputs" in dirnames:            # never descend into our own results
                dirnames.remove("outputs")
    raise FileNotFoundError(
        f"No folder containing {genres} found. Folders seen (first 25):\n  " + "\n  ".join(seen) +
        "\nPass --data-root <folder that contains classical/, jazz/, metal/, pop/>")


def list_wavs(root, genre):
    d = os.path.join(root, genre)
    return [os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith(".wav")]


def song_split(song_genre, val_frac=0.15, test_frac=0.15, seed=42, n_folds=0, fold=0):
    """Stratified split on SONGS (never on segments). Returns (train, val, test) song indices."""
    idx = np.arange(len(song_genre))
    song_genre = np.asarray(song_genre)
    if n_folds and n_folds > 1:
        skf = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
        rest, test = list(skf.split(idx, song_genre))[fold]
        tr, va = train_test_split(rest, test_size=val_frac, stratify=song_genre[rest], random_state=seed)
    else:
        rest, test = train_test_split(idx, test_size=test_frac, stratify=song_genre, random_state=seed)
        tr, va = train_test_split(rest, test_size=val_frac / (1 - test_frac),
                                  stratify=song_genre[rest], random_state=seed)
    return np.sort(tr), np.sort(va), np.sort(test)


def make_splits(X, y, song, mode="song", seed=42, n_folds=0, fold=0):
    """mode='random_segment' -> what the original repo did (leaky); mode='song' -> leak-free."""
    X, y, song = np.asarray(X), np.asarray(y), np.asarray(song)
    n = len(X)
    if mode == "random_segment":
        perm = np.random.RandomState(seed).permutation(n)
        a, b = int(0.75 * n), int(0.875 * n)          # 6000 / 1000 / 1000 for n = 8000
        parts = {"train": perm[:a], "val": perm[a:b], "test": perm[b:]}
    elif mode == "song":
        n_songs = song.max() + 1
        sg = np.zeros(n_songs, dtype=int)
        sg[song] = y
        tr, va, te = song_split(sg, seed=seed, n_folds=n_folds, fold=fold)
        parts = {k: np.where(np.isin(song, s))[0] for k, s in zip(("train", "val", "test"), (tr, va, te))}
    else:
        raise ValueError(mode)
    return {k: (X[i], y[i], song[i]) for k, i in parts.items()}


def leakage(splits):
    """Fraction of test segments whose parent song also appears in the training set."""
    tr_songs = np.unique(splits["train"][2])
    return float(np.isin(splits["test"][2], tr_songs).mean())

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/data_paper.py
"""Exact preprocessing of the original repo (softmaxautoencoder.py::load_data).

For each 30 s GTZAN clip: librosa.load (22.05 kHz, mono) -> first 600 000 samples -> reshape(15000, 40)
-> mean over the 40 axis (average pooling, pool=40 => 'artificial 551 Hz') -> 20 consecutive
500-sample segments. 400 songs x 20 = 8000 examples in R^500.
"""
import os
import warnings
import numpy as np

from .config import GENRES
from .utils import list_wavs, make_splits, leakage

N_SAMPLES, POOL, SEG_LEN, N_SEG = 600_000, 40, 500, 20


def load_paper_dataset(root, genres=GENRES, cache=None):
    if cache and os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        if d["X"].size:                                   # ignore empty/corrupt caches
            return d["X"], d["y"], d["song"], list(d["names"])
    import librosa
    X, y, song, names = [], [], [], []
    for gi, g in enumerate(genres):
        for path in list_wavs(root, g):
            try:
                audio, _ = librosa.load(path)               # defaults sr=22050, mono (as original)
            except Exception as e:                           # e.g. jazz.00054.wav is corrupt in the Kaggle copy
                warnings.warn(f"skipping unreadable file {path}: {e}")
                continue
            audio = audio[:N_SAMPLES]
            if len(audio) < N_SAMPLES:
                audio = np.pad(audio, (0, N_SAMPLES - len(audio)))
            pooled = audio.reshape(N_SAMPLES // POOL, POOL).mean(axis=1)
            sid = len(names)
            names.append(os.path.basename(path))
            for j in range(N_SEG):
                X.append(pooled[SEG_LEN * j: SEG_LEN * (j + 1)])
                y.append(gi)
                song.append(sid)
    if not names:
        raise RuntimeError(f"No .wav files could be read under {root}. Is this the genres_original folder?")
    X = np.asarray(X, dtype=np.float32)
    y, song = np.asarray(y, dtype=np.int32), np.asarray(song, dtype=np.int32)
    if cache:
        os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
        np.savez(cache, X=X, y=y, song=song, names=np.asarray(names))
    return X, y, song, names


def prepare_paper(root, cache=None, split="random_segment", seed=42, n_folds=0, fold=0):
    X, y, song, names = load_paper_dataset(root, cache=cache)
    splits = make_splits(X, y, song, mode=split, seed=seed, n_folds=n_folds, fold=fold)
    print(f"[data] {len(names)} songs -> {len(X)} segments | split={split} | "
          f"test-segments whose song is also in train: {leakage(splits):.1%}")
    return splits

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/data_mel.py
"""Improved front-end: log-mel spectrogram segments (3 s, 50 % overlap) instead of 551 Hz average-pooled waveform."""
import os
import warnings
import numpy as np

from .config import GENRES
from .utils import list_wavs, make_splits, leakage

SR, SEG_SEC, HOP_SEC = 22050, 3.0, 1.5
N_FFT, HOP, N_MELS, N_FRAMES = 1024, 512, 64, 128


def song_to_segments(path, librosa):
    y, _ = librosa.load(path, sr=SR, mono=True, duration=30.0)
    seg, step = int(SR * SEG_SEC), int(SR * HOP_SEC)
    out = []
    for start in range(0, max(len(y) - seg, 0) + 1, step):
        chunk = y[start:start + seg]
        if len(chunk) < seg:
            chunk = np.pad(chunk, (0, seg - len(chunk)))
        m = librosa.feature.melspectrogram(y=chunk, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, fmin=20)
        m = librosa.power_to_db(m, ref=1.0, top_db=80.0)      # absolute level kept (loudness is a genre cue)
        out.append(m[:, :N_FRAMES].astype(np.float32))        # (64, 130) -> (64, 128)
    return out


def load_mel_dataset(root, genres=GENRES, cache=None):
    if cache and os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        if d["X"].size:                                   # ignore empty/corrupt caches
            return d["X"], d["y"], d["song"], list(d["names"])
    import librosa
    X, y, song, names = [], [], [], []
    for gi, g in enumerate(genres):
        for path in list_wavs(root, g):
            try:
                segs = song_to_segments(path, librosa)
            except Exception as e:
                warnings.warn(f"skipping unreadable file {path}: {e}")
                continue
            sid = len(names)
            names.append(os.path.basename(path))
            for s in segs:
                X.append(s)
                y.append(gi)
                song.append(sid)
    if not names:
        raise RuntimeError(f"No .wav files could be read under {root}. Is this the genres_original folder?")
    X = np.asarray(X, dtype=np.float32)
    y, song = np.asarray(y, dtype=np.int32), np.asarray(song, dtype=np.int32)
    if cache:
        os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
        np.savez(cache, X=X, y=y, song=song, names=np.asarray(names))
    return X, y, song, names


def prepare_mel(root, cache=None, split="song", seed=42, n_folds=0, fold=0):
    X, y, song, names = load_mel_dataset(root, cache=cache)
    splits = make_splits(X, y, song, mode=split, seed=seed, n_folds=n_folds, fold=fold)
    mu, sd = splits["train"][0].mean(), splits["train"][0].std() + 1e-6      # train-only statistics
    splits = {k: (((a - mu) / sd)[..., None].astype(np.float32), yy, ss) for k, (a, yy, ss) in splits.items()}
    print(f"[data] {len(names)} songs -> {len(X)} segments | split={split} | "
          f"test-segments whose song is also in train: {leakage(splits):.1%}")
    return splits

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/evaluate.py
import os
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.decomposition import PCA

from .config import GENRES, GENRE_COLORS


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def song_level(probs, y, song):
    """Average segment probabilities per song -> one prediction per song."""
    ids = np.unique(song)
    P = np.stack([probs[song == i].mean(axis=0) for i in ids])
    T = np.array([y[song == i][0] for i in ids])
    return T, P.argmax(1)


def summarize(logits, y, song, genres=GENRES, name=""):
    probs = softmax(logits)
    pred = probs.argmax(1)
    st, sp = song_level(probs, y, song)
    rep = classification_report(y, pred, target_names=genres, output_dict=True, zero_division=0)
    res = {"segment_acc": float(accuracy_score(y, pred)),
           "segment_macro_f1": float(f1_score(y, pred, average="macro")),
           "song_acc": float(accuracy_score(st, sp)),
           "n_segments": int(len(y)), "n_songs": int(len(st)),
           "per_class": {g: {k: float(rep[g][k]) for k in ("precision", "recall", "f1-score")} for g in genres}}
    print(f"\n== {name} ==  segment acc {res['segment_acc']:.3f} | macro-F1 {res['segment_macro_f1']:.3f} | "
          f"song acc {res['song_acc']:.3f} ({res['n_songs']} songs)")
    print(classification_report(y, pred, target_names=genres, digits=3, zero_division=0))
    return res, pred


def plot_confusion(y, pred, path, title="Normalized confusion matrix", genres=GENRES):
    cm = confusion_matrix(y, pred, labels=range(len(genres))).astype(float)
    cm = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(genres)))
    ax.set_xticklabels(genres, rotation=45)
    ax.set_yticks(range(len(genres)))
    ax.set_yticklabels(genres)
    for i in range(len(genres)):
        for j in range(len(genres)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", color="white" if cm[i, j] > .5 else "black")
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label"); ax.set_title(title)
    fig.colorbar(im); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def plot_pca(Z, y, title, path, genres=GENRES):
    Z2 = PCA(n_components=2).fit_transform(Z.reshape(len(Z), -1))
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    for k in reversed(range(len(genres))):                      # same draw order as original plots
        m = y == k
        ax.scatter(Z2[m, 0], Z2[m, 1], c=GENRE_COLORS[k], alpha=.5, edgecolor="k", s=18, label=genres[k].capitalize())
    ax.set_title(title); ax.legend(loc="upper right")
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def plot_history(hist, path, title="Training curves"):
    keys = [k for k in hist if hist[k]]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for k in keys:
        (axes[0] if "loss" in k else axes[1]).plot(hist[k], label=k)
    axes[0].set_title("loss"); axes[1].set_title("accuracy")
    for a in axes:
        a.legend(fontsize=7); a.set_xlabel("epoch")
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def save_json(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/models_paper.py
"""Exact architectures from the original repo, ported from TF1 (tf.contrib / placeholders) to tf.keras.

softmaxautoencoder.py  -> build_dsae        (Deep Softmax Autoencoder, the paper's final model)
encoder.py             -> build_vanilla_ae  (reconstruction-only autoencoder, 400 epochs in the repo)
tfsoftmax.py           -> build_baseline_nn (500 -> 128 tanh -> 4)

Faithfulness notes
* Xavier-uniform init == Keras glorot_uniform (default); biases start at zero (default).
* Original dropout used keep_prob=0.8  ->  rate 0.2, applied after the 2nd encoder layer and 2nd decoder layer only.
* Original decoder ends with ReLU (so it can never output negative samples although the waveform is in [-1, 1]).
  Kept as default for reproduction; `recon_activation="linear"` is offered for the tuned variant.
* Original loss (code, not the paper's Eq. 3):  mean((x - x_hat)^2) + 0.1 * mean(softmax_cross_entropy)
"""
from tensorflow.keras import layers, Model, optimizers, losses


def _encode(x, latent, dropout):
    x = layers.Dense(256, activation="sigmoid", name="enc1")(x)
    x = layers.Dense(192, activation="tanh", name="enc2")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation="relu", name="enc3")(x)
    return layers.Dense(latent, activation="relu", name="embedding")(x)


def _decode(z, input_dim, dropout, recon_activation):
    x = layers.Dense(128, activation="sigmoid", name="dec1")(z)
    x = layers.Dense(192, activation="sigmoid", name="dec2")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(256, activation="relu", name="dec3")(x)
    return layers.Dense(input_dim, activation=recon_activation, name="recon")(x)


def build_baseline_nn(input_dim=500, n_classes=4, lr=1e-4, hidden=128):
    inp = layers.Input((input_dim,), name="x")
    h = layers.Dense(hidden, activation="tanh", name="hidden")(inp)
    logits = layers.Dense(n_classes, name="logits")(h)
    m = Model(inp, logits, name="baseline_nn")
    m.compile(optimizers.Adam(lr), loss=losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["sparse_categorical_accuracy"])
    return m


def build_vanilla_ae(input_dim=500, latent=64, dropout=0.2, lr=1e-4, recon_activation="relu"):
    inp = layers.Input((input_dim,), name="x")
    z = _encode(inp, latent, dropout)
    recon = _decode(z, input_dim, dropout, recon_activation)
    m = Model(inp, recon, name="vanilla_ae")
    m.compile(optimizers.Adam(lr), loss="mse")
    return m


def build_dsae(input_dim=500, latent=64, n_classes=4, dropout=0.2, lr=1e-4,
               recon_weight=1.0, cls_weight=0.1, recon_activation="relu"):
    inp = layers.Input((input_dim,), name="x")
    z = _encode(inp, latent, dropout)
    recon = _decode(z, input_dim, dropout, recon_activation)
    c = layers.Dense(32, activation="tanh", name="cls1")(z)
    c = layers.Dense(16, activation="tanh", name="cls2")(c)
    logits = layers.Dense(n_classes, name="logits")(c)
    m = Model(inp, {"recon": recon, "logits": logits}, name="deep_softmax_autoencoder")
    m.compile(optimizers.Adam(lr),
              loss={"recon": "mse", "logits": losses.SparseCategoricalCrossentropy(from_logits=True)},
              loss_weights={"recon": recon_weight, "logits": cls_weight},
              metrics={"logits": ["sparse_categorical_accuracy"]})
    return m

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/models_improved.py
"""Better front-end + back-end:  log-mel (64 x 128 x 1) -> small VGG-style CNN.

build_cnn        : pure classifier, 64-d 'embedding' layer before the softmax.
build_conv_dsae  : the paper's idea (autoencoder + classifier on the SAME 64-d code) but convolutional.
                   Trained as a *denoising* AE when SpecAugment is on (input masked, target clean).
"""
from tensorflow.keras import layers, Model, optimizers, losses


def _conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)


def build_cnn(input_shape=(64, 128, 1), n_classes=4, base=32, latent=64, dropout=0.3, lr=1e-3, **_):
    inp = layers.Input(input_shape, name="x")
    x = inp
    for f in (base, 2 * base, 4 * base, 4 * base):
        x = _conv_block(x, f)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(dropout / 2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    z = layers.Dense(latent, activation="relu", name="embedding")(x)
    z = layers.Dropout(dropout)(z)
    logits = layers.Dense(n_classes, name="logits")(z)
    m = Model(inp, logits, name="mel_cnn")
    m.compile(optimizers.Adam(lr), loss=losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["sparse_categorical_accuracy"])
    return m


def build_conv_dsae(input_shape=(64, 128, 1), n_classes=4, base=32, latent=64, dropout=0.3, lr=1e-3,
                    recon_weight=1.0, cls_weight=1.0, **_):
    inp = layers.Input(input_shape, name="x")
    x = inp
    for f in (base, 2 * base, 4 * base, 4 * base):          # 64x128 -> 4x8
        x = _conv_block(x, f)
        x = layers.MaxPooling2D(2)(x)
    h, w, c = input_shape[0] // 16, input_shape[1] // 16, 4 * base
    x = layers.Flatten()(x)
    x = layers.Dropout(dropout)(x)
    z = layers.Dense(latent, activation="relu", name="embedding")(x)
    # classifier head: same shape as the paper (64 -> 32 -> 16 -> classes, tanh)
    k = layers.Dense(32, activation="tanh")(z)
    k = layers.Dense(16, activation="tanh")(k)
    logits = layers.Dense(n_classes, name="logits")(k)
    # decoder
    d = layers.Dense(h * w * c, activation="relu")(z)
    d = layers.Reshape((h, w, c))(d)
    for f in (4 * base, 2 * base, base, max(base // 2, 8)):  # 4x8 -> 64x128
        d = layers.UpSampling2D(2)(d)
        d = _conv_block(d, f)
    recon = layers.Conv2D(1, 3, padding="same", name="recon")(d)
    m = Model(inp, {"recon": recon, "logits": logits}, name="conv_deep_softmax_autoencoder")
    m.compile(optimizers.Adam(lr),
              loss={"recon": "mse", "logits": losses.SparseCategoricalCrossentropy(from_logits=True)},
              loss_weights={"recon": recon_weight, "logits": cls_weight},
              metrics={"logits": ["sparse_categorical_accuracy"]})
    return m

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/training.py
import numpy as np
import tensorflow as tf
from tensorflow import keras


def predict_logits(model, X, bs=256):
    out = model.predict(X, batch_size=bs, verbose=0)
    if isinstance(out, dict):
        return out["logits"]
    if isinstance(out, (list, tuple)):
        return out[model.output_names.index("logits")]
    return out


def spec_augment(x, max_f=10, max_t=16):
    """One frequency mask + one time mask (zero == train mean because inputs are standardised)."""
    n_mels, n_t = x.shape[0], x.shape[1]
    f = tf.random.uniform([], 0, max_f + 1, tf.int32)
    f0 = tf.random.uniform([], 0, n_mels - max_f, tf.int32)
    t = tf.random.uniform([], 0, max_t + 1, tf.int32)
    t0 = tf.random.uniform([], 0, n_t - max_t, tf.int32)
    fi, ti = tf.range(n_mels), tf.range(n_t)
    fm = tf.cast(tf.logical_not((fi >= f0) & (fi < f0 + f)), x.dtype)
    tm = tf.cast(tf.logical_not((ti >= t0) & (ti < t0 + t)), x.dtype)
    return x * fm[:, None, None] * tm[None, :, None]


def make_dataset(X, y, batch, augment=False, ae=False, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y.astype(np.int32)))
    if shuffle:
        ds = ds.shuffle(min(len(X), 20000), reshuffle_each_iteration=True)

    def f(x, yy):
        xi = spec_augment(x) if augment else x
        return (xi, {"recon": x, "logits": yy}) if ae else (xi, yy)

    return ds.map(f, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)


class BestValAcc(keras.callbacks.Callback):
    """Tracks validation accuracy itself (metric names differ between Keras 2/3), keeps best weights, early-stops."""

    def __init__(self, Xva, yva, patience=10):
        super().__init__()
        self.Xva, self.yva, self.patience = Xva, yva, patience
        self.accs, self.best, self.best_w, self.wait = [], -1.0, None, 0

    def on_epoch_end(self, epoch, logs=None):
        acc = float((predict_logits(self.model, self.Xva).argmax(1) == self.yva).mean())
        self.accs.append(acc)
        if acc > self.best:
            self.best, self.best_w, self.wait = acc, self.model.get_weights(), 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_w is not None:
            self.model.set_weights(self.best_w)


def fit_model(model, Xtr, ytr, Xva, yva, ae, epochs, batch, augment=False, patience=10, lr_patience=4, verbose=0):
    ds = make_dataset(Xtr, ytr, batch, augment=augment, ae=ae)
    val = (Xva, {"recon": Xva, "logits": yva}) if ae else (Xva, yva)
    cb = BestValAcc(Xva, yva, patience)
    cbs = [cb, keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=lr_patience, min_lr=1e-6)]
    h = model.fit(ds, validation_data=val, epochs=epochs, callbacks=cbs, verbose=verbose)
    hist = {k: v for k, v in h.history.items() if "loss" in k or "accuracy" in k}
    hist["val_seg_acc"] = cb.accs
    return hist, cb.best

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/src/genre/experiment.py
"""One function used by both the training script and the Optuna tuner."""
import numpy as np
import tensorflow as tf

from . import models_improved as MI, models_paper as MP
from .evaluate import summarize
from .training import fit_model, predict_logits
from .utils import set_seed

AE_KINDS = {"conv_dsae", "paper_dsae"}


def build_model(kind, p, input_shape):
    if kind == "cnn":
        return MI.build_cnn(input_shape, **p)
    if kind == "conv_dsae":
        return MI.build_conv_dsae(input_shape, **p)
    if kind == "paper_dsae":
        return MP.build_dsae(input_dim=input_shape[0], **p)
    if kind == "paper_nn":
        return MP.build_baseline_nn(input_dim=input_shape[0], **p)
    raise ValueError(kind)


def train_eval(kind, params, data, epochs, batch=64, augment=False, patience=10, seed=42, verbose=0, eval_test=True):
    """params -> model kwargs (lr, base, latent, dropout, cls_weight ...). Returns dict with metrics + models."""
    tf.keras.backend.clear_session()
    set_seed(seed)
    (Xtr, ytr, _), (Xva, yva, sva), (Xte, yte, ste) = data["train"], data["val"], data["test"]
    ae = kind in AE_KINDS
    model = build_model(kind, params, Xtr.shape[1:])
    hist, best_val_seg = fit_model(model, Xtr, ytr, Xva, yva, ae, epochs, batch,
                                   augment=augment and Xtr.ndim == 4, patience=patience, verbose=verbose)
    val_res, _ = summarize(predict_logits(model, Xva), yva, sva, name=f"{kind} / val")
    if eval_test:                       # the tuner passes False: test data is never touched while searching
        test_res, test_pred = summarize(predict_logits(model, Xte), yte, ste, name=f"{kind} / TEST")
    else:
        test_res, test_pred = None, None
    return {"model": model, "history": hist, "val": val_res, "test": test_res, "test_pred": test_pred,
            "best_val_segment_acc": best_val_seg}


def embeddings(model, X):
    try:
        emb = tf.keras.Model(model.input, model.get_layer("embedding").output)
        return emb.predict(X, batch_size=256, verbose=0)
    except ValueError:
        return predict_logits(model, X)

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/scripts/01_reproduce_paper.py
"""Reproduce the paper / repo: baseline NN, vanilla AE (400 ep), Deep Softmax Autoencoder (200 ep), Tables 1-2, Figs 1,4,5,6.

  python scripts/01_reproduce_paper.py --data-root auto --out outputs/paper
  python scripts/01_reproduce_paper.py --split song        # same models, leak-free song-level split (honest numbers)
  python scripts/01_reproduce_paper.py --quick             # 3-epoch smoke test
"""
import argparse, os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
import numpy as np
import tensorflow as tf

from genre.data_paper import prepare_paper
from genre.evaluate import plot_confusion, plot_history, plot_pca, save_json, summarize
from genre.models_paper import build_baseline_nn, build_dsae, build_vanilla_ae
from genre.training import predict_logits
from genre.utils import find_gtzan_root, set_seed


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="auto")
    ap.add_argument("--out", default="outputs/paper")
    ap.add_argument("--split", default="random_segment", choices=["random_segment", "song"])
    ap.add_argument("--epochs", type=int, default=200)       # baseline + DSAE (repo: num_epochs=200)
    ap.add_argument("--ae-epochs", type=int, default=400)    # vanilla AE (repo encoder.py: num_epochs=400)
    ap.add_argument("--batch", type=int, default=512)
    ap.add_argument("--cls-weight", type=float, default=0.1)  # repo: classificationweight = 0.1
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--quick", action="store_true")
    a = ap.parse_args()
    if a.quick:
        a.epochs = a.ae_epochs = 3
    os.makedirs(a.out, exist_ok=True)
    set_seed(a.seed)

    root = find_gtzan_root(a.data_root)
    print("GTZAN root:", root)
    splits = prepare_paper(root, cache=os.path.join(os.path.dirname(a.out.rstrip("/")) or ".", "cache", "paper_500d.npz"),
                           split=a.split, seed=a.seed)
    (Xtr, ytr, _), (Xva, yva, sva), (Xte, yte, ste) = splits["train"], splits["val"], splits["test"]
    perm = np.random.RandomState(a.seed).permutation(len(Xtr))        # shuffle ONCE like the repo (fit(shuffle=False))
    Xtr, ytr = Xtr[perm], ytr[perm]
    common = dict(batch_size=a.batch, epochs=a.epochs, shuffle=False, verbose=0)
    results, hists = {}, {}

    # 1) two-layer baseline (tfsoftmax.py)
    base = build_baseline_nn()
    h = base.fit(Xtr, ytr, validation_data=(Xva, yva), **common)
    hists["baseline"] = h.history

    # 2) vanilla autoencoder (encoder.py)
    vae = build_vanilla_ae()
    h = vae.fit(Xtr, Xtr, validation_data=(Xva, Xva), batch_size=a.batch, epochs=a.ae_epochs, shuffle=False, verbose=0)
    hists["vanilla_ae"] = h.history

    # 3) Deep Softmax Autoencoder (softmaxautoencoder.py)
    dsae = build_dsae(cls_weight=a.cls_weight)
    h = dsae.fit(Xtr, {"recon": Xtr, "logits": ytr}, validation_data=(Xva, {"recon": Xva, "logits": yva}), **common)
    hists["dsae"] = {k: v for k, v in h.history.items() if "loss" in k or "accuracy" in k}

    # ---- Table 1: accuracies
    acc = lambda m, X, y: float((predict_logits(m, X).argmax(1) == y).mean())
    table1 = {"two_layer_nn": {s: acc(base, X, y) for s, (X, y) in {"train": (Xtr, ytr), "dev": (Xva, yva), "test": (Xte, yte)}.items()},
              "deep_softmax_ae": {s: acc(dsae, X, y) for s, (X, y) in {"train": (Xtr, ytr), "dev": (Xva, yva), "test": (Xte, yte)}.items()}}
    print("\nTable 1 (classification accuracy)  paper: NN 52.0/38.1/36.4  DSAE 94.9/64.1/65.3")
    for k, v in table1.items():
        print(f"  {k:18s} train {v['train']:.3f}  dev {v['dev']:.3f}  test {v['test']:.3f}")

    # ---- Table 2 + Fig 4 : per-class metrics, confusion matrix on test
    res, pred = summarize(predict_logits(dsae, Xte), yte, ste, name="Deep Softmax AE / test (Table 2)")
    plot_confusion(yte, pred, os.path.join(a.out, "fig4_confusion_dsae_test.png"))
    results.update(table1=table1, table2_dsae_test=res)

    # ---- Figs 1,5,6: PCA visualisations on dev+test (2000 examples, as in the repo's *_dev.csv)
    Xv, yv = np.concatenate([Xva, Xte]), np.concatenate([yva, yte])
    plot_pca(Xv, yv, "PCA of 500-Dim Raw Audio Data", os.path.join(a.out, "fig1_pca_raw.png"))
    plot_pca(tf.keras.Model(vae.input, vae.get_layer("embedding").output).predict(Xv, verbose=0), yv,
             "PCA of 64-Dim Vanilla Autoencoder Bottleneck", os.path.join(a.out, "fig5a_pca_vanilla_ae.png"))
    plot_pca(tf.keras.Model(dsae.input, dsae.get_layer("embedding").output).predict(Xv, verbose=0), yv,
             "PCA of 64-Dim Deep Softmax AE Bottleneck", os.path.join(a.out, "fig5b_pca_dsae_64.png"))
    plot_pca(predict_logits(dsae, Xv), yv, "PCA of 4-Dim Deep Softmax AE Logits", os.path.join(a.out, "fig6b_pca_dsae_logits.png"))
    plot_pca(predict_logits(base, Xv), yv, "PCA of 4-Dim Two Layer NN Logits", os.path.join(a.out, "fig6a_pca_nn_logits.png"))
    plot_history(hists["dsae"], os.path.join(a.out, "fig3_dsae_curves.png"), "Deep Softmax AE")

    save_json(results, os.path.join(a.out, "metrics.json"))
    print("\nSaved figures + metrics.json to", a.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/scripts/02_train_improved.py
"""Train the improved models on log-mel spectrograms with a leak-free SONG-level split.

  python scripts/02_train_improved.py --model cnn --out outputs/cnn
  python scripts/02_train_improved.py --model conv_dsae --out outputs/conv_dsae
  python scripts/02_train_improved.py --model conv_dsae --folds 5 --fold 0      # 5-fold CV (run folds 0..4)
  python scripts/02_train_improved.py --model conv_dsae --params outputs/tune/best_params.json   # tuned
"""
import argparse, json, os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
import numpy as np

from genre.data_mel import prepare_mel
from genre.evaluate import plot_confusion, plot_history, plot_pca, save_json
from genre.experiment import embeddings, train_eval
from genre.utils import find_gtzan_root, set_seed


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="conv_dsae", choices=["cnn", "conv_dsae"])
    ap.add_argument("--data-root", default="auto")
    ap.add_argument("--out", default="outputs/improved")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--base", type=int, default=32)
    ap.add_argument("--latent", type=int, default=64)
    ap.add_argument("--dropout", type=float, default=0.3)
    ap.add_argument("--cls-weight", type=float, default=1.0)
    ap.add_argument("--recon-weight", type=float, default=1.0)
    ap.add_argument("--no-augment", action="store_true")
    ap.add_argument("--patience", type=int, default=12)
    ap.add_argument("--folds", type=int, default=0)
    ap.add_argument("--fold", type=int, default=0)
    ap.add_argument("--params", default=None, help="JSON from 03_tune_optuna.py (overrides the flags above)")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--quick", action="store_true")
    a = ap.parse_args()
    if a.quick:
        a.epochs = 2
    os.makedirs(a.out, exist_ok=True)
    set_seed(a.seed)

    p = dict(lr=a.lr, base=a.base, latent=a.latent, dropout=a.dropout)
    if a.model == "conv_dsae":
        p.update(cls_weight=a.cls_weight, recon_weight=a.recon_weight)
    batch, augment = a.batch, not a.no_augment
    if a.params:
        tuned = json.load(open(a.params))
        batch, augment = tuned.pop("batch", batch), tuned.pop("spec_aug", augment)
        p.update({k: v for k, v in tuned.items() if k in ("lr", "base", "latent", "dropout", "cls_weight", "recon_weight")})
    print("hyper-parameters:", p, "| batch", batch, "| spec_aug", augment)

    root = find_gtzan_root(a.data_root)
    cache = os.path.join(os.path.dirname(a.out.rstrip("/")) or ".", "cache", "mel64x128.npz")
    data = prepare_mel(root, cache=cache, split="song", seed=a.seed, n_folds=a.folds, fold=a.fold)

    r = train_eval(a.model, p, data, epochs=a.epochs, batch=batch, augment=augment, patience=a.patience, seed=a.seed, verbose=2)
    Xte, yte, _ = data["test"]
    plot_confusion(yte, r["test_pred"], os.path.join(a.out, "confusion_test.png"), f"{a.model}: test (song-level split)")
    plot_pca(embeddings(r["model"], Xte), yte, f"PCA of 64-d {a.model} embedding (test songs)", os.path.join(a.out, "pca_embedding_test.png"))
    plot_history(r["history"], os.path.join(a.out, "curves.png"), a.model)
    save_json({"model": a.model, "params": p, "batch": batch, "spec_aug": augment, "fold": a.fold, "folds": a.folds,
               "val": r["val"], "test": r["test"]}, os.path.join(a.out, "metrics.json"))
    try:
        r["model"].save_weights(os.path.join(a.out, f"{a.model}.weights.h5"))
    except Exception as e:
        print("could not save weights:", e)
    print(f"\nFINAL TEST  segment acc {r['test']['segment_acc']:.3f} | song acc {r['test']['song_acc']:.3f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/scripts/03_tune_optuna.py
"""Hyper-parameter search (Optuna / TPE) with a leak-free song-level split. Objective = validation SONG accuracy.

  python scripts/03_tune_optuna.py --model conv_dsae --trials 20 --epochs 30
  python scripts/03_tune_optuna.py --model cnn       --trials 20 --epochs 30
  python scripts/03_tune_optuna.py --model paper_dsae --trials 30 --epochs 120   # tunes the ORIGINAL 500-d MLP model
"""
import argparse, os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
try:
    import optuna
except ImportError:
    sys.exit("pip install optuna")

from genre.data_mel import prepare_mel
from genre.data_paper import prepare_paper
from genre.evaluate import save_json
from genre.experiment import train_eval
from genre.utils import find_gtzan_root, set_seed


def suggest(trial, kind):
    if kind in ("cnn", "conv_dsae"):
        p = dict(lr=trial.suggest_float("lr", 3e-4, 3e-3, log=True),
                 base=trial.suggest_categorical("base", [16, 32, 48]),
                 latent=trial.suggest_categorical("latent", [32, 64, 128]),
                 dropout=trial.suggest_float("dropout", 0.1, 0.5))
        if kind == "conv_dsae":
            p["cls_weight"] = trial.suggest_float("cls_weight", 0.1, 5.0, log=True)
        extra = dict(batch=trial.suggest_categorical("batch", [32, 64, 128]),
                     spec_aug=trial.suggest_categorical("spec_aug", [True, False]))
    else:   # paper_dsae (500-d raw-audio MLP)
        p = dict(lr=trial.suggest_float("lr", 1e-4, 3e-3, log=True),
                 latent=trial.suggest_categorical("latent", [32, 64, 128]),
                 dropout=trial.suggest_float("dropout", 0.0, 0.5),
                 cls_weight=trial.suggest_float("cls_weight", 0.05, 5.0, log=True),
                 recon_activation=trial.suggest_categorical("recon_activation", ["relu", "linear"]))
        extra = dict(batch=trial.suggest_categorical("batch", [128, 256, 512]), spec_aug=False)
    return p, extra


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="conv_dsae", choices=["cnn", "conv_dsae", "paper_dsae"])
    ap.add_argument("--data-root", default="auto")
    ap.add_argument("--out", default="outputs/tune")
    ap.add_argument("--trials", type=int, default=20)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--final-epochs", type=int, default=80)
    ap.add_argument("--patience", type=int, default=8)
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    set_seed(a.seed)
    root = find_gtzan_root(a.data_root)
    cdir = os.path.join(os.path.dirname(a.out.rstrip("/")) or ".", "cache")
    if a.model == "paper_dsae":
        data = prepare_paper(root, cache=os.path.join(cdir, "paper_500d.npz"), split="song", seed=a.seed)
    else:
        data = prepare_mel(root, cache=os.path.join(cdir, "mel64x128.npz"), split="song", seed=a.seed)

    def objective(trial):
        p, extra = suggest(trial, a.model)
        r = train_eval(a.model, p, data, epochs=a.epochs, batch=extra["batch"], augment=extra["spec_aug"],
                       patience=a.patience, seed=a.seed, eval_test=False)
        trial.set_user_attr("val_segment_acc", r["val"]["segment_acc"])
        return r["val"]["song_acc"]

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=a.seed))
    study.optimize(objective, n_trials=a.trials)
    best = dict(study.best_params)
    print("\nBEST val song-acc", study.best_value, best)
    save_json(best, os.path.join(a.out, "best_params.json"))
    study.trials_dataframe().to_csv(os.path.join(a.out, "trials.csv"), index=False)

    # retrain with the best config for longer, report TEST once
    p, extra = {}, {}
    for k, v in best.items():
        (extra if k in ("batch", "spec_aug") else p)[k] = v
    r = train_eval(a.model, p, data, epochs=a.final_epochs, batch=extra.get("batch", 64),
                   augment=extra.get("spec_aug", False), patience=12, seed=a.seed, verbose=0)
    save_json({"best_params": best, "val": r["val"], "test": r["test"]}, os.path.join(a.out, "final_metrics.json"))
    print(f"\nFINAL (best params) TEST segment acc {r['test']['segment_acc']:.3f} | song acc {r['test']['song_acc']:.3f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/genre-extraction-repro/scripts/04_demo.py
"""Live demo of the trained mel-CNN (no training, takes seconds once features are cached).

  python scripts/04_demo.py --weights outputs/cnn/cnn.weights.h5 --n 10
  python scripts/04_demo.py --weights outputs/cnn/cnn.weights.h5 --audio my_song.wav other.wav

What it does
 1. Re-creates the exact song-level split (seed 42) and the train-only standardisation.
 2. Loads the weights and RE-COMPUTES accuracy on the 60 unseen test songs; compares with the saved metrics.json.
 3. Shows N random test songs: true genre, predicted genre, confidence.
 4. For every --audio file: predicts its genre and saves a picture (spectrogram + probabilities).
"""
import argparse, json, os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from genre.config import GENRES
from genre.data_mel import load_mel_dataset, song_to_segments
from genre.evaluate import softmax
from genre.utils import find_gtzan_root, make_splits


def locate(path, filename):
    """Return `path` if it exists, otherwise search /kaggle/input, /kaggle/working and . for `filename`."""
    if path and os.path.exists(path):
        return path
    for base in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.isdir(base):
            continue
        for dirpath, _, files in os.walk(base):
            if filename in files:
                found = os.path.join(dirpath, filename)
                print(f"[locate] {filename} -> {found}")
                return found
    raise FileNotFoundError(f"{filename} not found. Attach the notebook output / dataset that contains it "
                            f"(Add Input), or train it first (scripts/02_train_improved.py --model cnn).")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", default="outputs/cnn/cnn.weights.h5")
    ap.add_argument("--data-root", default="auto")
    ap.add_argument("--cache", default="outputs/cache/mel64x128.npz")
    ap.add_argument("--audio", nargs="*", default=[])
    ap.add_argument("--n", type=int, default=10)
    ap.add_argument("--out", default="outputs/demo")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    # 1) same data, same split, same normalisation as training
    a.weights = locate(a.weights, os.path.basename(a.weights))
    cache = a.cache
    if not os.path.exists(cache):
        try:
            cache = locate(cache, os.path.basename(cache))
        except FileNotFoundError:
            pass                                   # fall back to rebuilding features from the audio
    root = None if os.path.exists(cache) else find_gtzan_root(a.data_root)   # audio dataset only needed without a cache
    X, y, song, names = load_mel_dataset(root, cache=cache)
    sp = make_splits(X, y, song, mode="song", seed=a.seed)
    mu, sd = sp["train"][0].mean(), sp["train"][0].std() + 1e-6
    Xte, yte, ste = sp["test"]
    Xte = ((Xte - mu) / sd)[..., None].astype(np.float32)

    # 2) model
    from genre.models_improved import build_cnn
    model = build_cnn((64, 128, 1))
    model.load_weights(a.weights)
    print(f"loaded {a.weights}\n")

    probs = softmax(model.predict(Xte, batch_size=256, verbose=0))
    seg_acc = float((probs.argmax(1) == yte).mean())
    ids = np.unique(ste)
    P = np.stack([probs[ste == i].mean(0) for i in ids])
    T = np.array([yte[ste == i][0] for i in ids])
    song_acc = float((P.argmax(1) == T).mean())
    print(f"Unseen TEST songs: {len(ids)} | segment accuracy {seg_acc:.3f} | song accuracy {song_acc:.3f}")
    mj = os.path.join(os.path.dirname(a.weights), "metrics.json")
    if os.path.exists(mj):
        saved = json.load(open(mj))["test"]
        print(f"saved in metrics.json:             segment accuracy {saved['segment_acc']:.3f} | song accuracy {saved['song_acc']:.3f}")

    # 3) a few songs, one by one
    rng = np.random.RandomState(0)
    pick = rng.choice(len(ids), size=min(a.n, len(ids)), replace=False)
    print(f"\n{'song file':<22}{'true':<11}{'predicted':<11}{'confidence':<11}")
    for k in pick:
        p = P[k]
        mark = "OK " if p.argmax() == T[k] else "WRONG"
        print(f"{names[ids[k]]:<22}{GENRES[T[k]]:<11}{GENRES[p.argmax()]:<11}{p.max():<11.1%}{mark}")

    # 4) your own audio
    for path in a.audio:
        segs = np.stack(song_to_segments(path, __import__("librosa")))
        x = ((segs - mu) / sd)[..., None].astype(np.float32)
        p = softmax(model.predict(x, verbose=0)).mean(0)
        print(f"\n{os.path.basename(path)}: {len(segs)} windows -> {GENRES[p.argmax()]} ({p.max():.1%})  | " +
              ", ".join(f"{g} {v:.1%}" for g, v in zip(GENRES, p)))
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
        ax[0].imshow(segs[0], origin="lower", aspect="auto"); ax[0].set_title("log-mel spectrogram (first 3 s)")
        ax[0].set_xlabel("time frames"); ax[0].set_ylabel("mel band")
        ax[1].bar(GENRES, p); ax[1].set_ylim(0, 1); ax[1].set_title(f"prediction: {GENRES[p.argmax()]} ({p.max():.0%})")
        fig.tight_layout()
        fp = os.path.join(a.out, "demo_" + os.path.splitext(os.path.basename(path))[0] + ".png")
        fig.savefig(fp, dpi=120); plt.close(fig)
        print("saved", fp)


if __name__ == "__main__":
    main()

## 2. Smoke test (≈ 1–2 min; also reveals the GTZAN folder that was auto-detected)

In [ ]:
!cd /kaggle/working/genre-extraction-repro && python scripts/01_reproduce_paper.py --quick --out outputs/smoke

## 3. Objective 1 — reproduce the paper
`random_segment` = original (leaky) protocol; `song` = same models, honest split.

In [ ]:
!cd /kaggle/working/genre-extraction-repro && python scripts/01_reproduce_paper.py --split random_segment --out outputs/paper_leaky

In [ ]:
!cd /kaggle/working/genre-extraction-repro && python scripts/01_reproduce_paper.py --split song --out outputs/paper_song

## 4. Objective 2 — better front-end (log-mel) + better models, song-level split

In [ ]:
!cd /kaggle/working/genre-extraction-repro && python scripts/02_train_improved.py --model cnn --out outputs/cnn

In [ ]:
!cd /kaggle/working/genre-extraction-repro && python scripts/02_train_improved.py --model conv_dsae --out outputs/conv_dsae

## 5. Objective 2 — hyper-parameter tuning (Optuna)
About 20–40 min on a T4 for 20 trials. Lower `--trials` if short on time.

In [ ]:
!cd /kaggle/working/genre-extraction-repro && python scripts/03_tune_optuna.py --model conv_dsae --trials 20 --epochs 30 --out outputs/tune

In [ ]:
!cd /kaggle/working/genre-extraction-repro && python scripts/02_train_improved.py --model conv_dsae --params outputs/tune/best_params.json --epochs 80 --out outputs/conv_dsae_tuned

## 6. Results

In [ ]:
import json, glob
from IPython.display import Image, display
for f in sorted(glob.glob("/kaggle/working/genre-extraction-repro/outputs/*/metrics.json")):
    m = json.load(open(f)); t = m.get("test") or m.get("table1")
    print(f.split("outputs/")[1], "->", json.dumps(t)[:300])
for p in ["paper_leaky/fig4_confusion_dsae_test.png", "paper_leaky/fig5b_pca_dsae_64.png",
          "conv_dsae/confusion_test.png", "conv_dsae/pca_embedding_test.png"]:
    fp = "/kaggle/working/genre-extraction-repro/outputs/" + p
    if os.path.exists(fp): display(Image(fp))

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/genre_outputs", "zip", "/kaggle/working/genre-extraction-repro/outputs")   # download from the Output tab

## 7. Live demo for your teacher (needs outputs/cnn/cnn.weights.h5 from section 4; takes seconds)
Re-computes accuracy on the 60 unseen test songs, shows individual songs, and can classify YOUR audio file.

In [ ]:
!cd /kaggle/working/genre-extraction-repro && python scripts/04_demo.py --weights outputs/cnn/cnn.weights.h5 --n 12

In [ ]:
# put your own .wav in the notebook (Add Input / Upload), then edit the path below
!cd /kaggle/working/genre-extraction-repro && python scripts/04_demo.py --weights outputs/cnn/cnn.weights.h5 --n 0 --audio /kaggle/input/YOUR_DATASET/my_song.wav

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('/kaggle/working/genre-extraction-repro/outputs/demo/*.png')): display(Image(f))